In [1]:
import os
import numpy as np
import pandas as pd

# -----------------------------------------------------------------------------
# 1. FILE CONFIGURATION & EFFICIENT LOADING
# -----------------------------------------------------------------------------
FILE_PATH = "D:\\STATS NZ DATASET\\household-labour-force-survey-population-rebase-September-2018-March-2025-quarters-csv.csv"  # Update with your exact file path

print(f"Loading dataset: {FILE_PATH} ...")
file_size_mb = os.path.getsize(FILE_PATH) / (1024 * 1024)
print(f"File Size: {file_size_mb:.2f} MB")

# Define all possible missing value representations common in Stats NZ files
NA_VALUES = [
    "n/a",
    "N/A",
    "NA",
    "null",
    "NULL",
    "None",
    "",
    " ",
    "..",
    "C",
    "S",
]

# Read CSV with explicit NA handling and optimized memory usage
df = pd.read_csv(
    FILE_PATH,
    na_values=NA_VALUES,
    keep_default_na=True,
    low_memory=False,  # Prevents mixed data type warnings on large datasets
)

print(
    f"Successfully loaded dataset with {df.shape[0]:,} rows and {df.shape[1]} columns.\n"
)


Loading dataset: D:\STATS NZ DATASET\household-labour-force-survey-population-rebase-September-2018-March-2025-quarters-csv.csv ...
File Size: 382.08 MB
Successfully loaded dataset with 1,210,757 rows and 58 columns.



In [2]:

# -----------------------------------------------------------------------------
# 2. GENERAL DATASET AUDIT (Shape, Memory, Duplicates)
# -----------------------------------------------------------------------------
total_rows = len(df)
duplicate_rows = df.duplicated().sum()
duplicate_pct = (duplicate_rows / total_rows) * 100
memory_usage_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)

print("=" * 80)
print("1. DATASET OVERVIEW AUDIT")
print("=" * 80)
print(f"Total Records (Rows)   : {total_rows:,}")
print(f"Total Features (Cols)  : {df.shape[1]}")
print(
    f"Duplicate Rows         : {duplicate_rows:,} ({duplicate_pct:.2f}% of total)"
)
print(f"Memory Allocation      : {memory_usage_mb:.2f} MB\n")



1. DATASET OVERVIEW AUDIT
Total Records (Rows)   : 1,210,757
Total Features (Cols)  : 58
Duplicate Rows         : 0 (0.00% of total)
Memory Allocation      : 2457.29 MB



In [3]:
# -----------------------------------------------------------------------------
# 3. COLUMN-LEVEL HYGIENE & VALIDATION AUDIT
# -----------------------------------------------------------------------------
column_audit = []

for col in df.columns:
    col_data = df[col]

    # Count missing/null values
    null_count = col_data.isna().sum()
    null_pct = (null_count / total_rows) * 100

    # Data type detected by pandas
    inferred_dtype = str(col_data.dtype)

    # Count distinct values
    unique_count = col_data.nunique(dropna=True)

    # Check for non-numeric/invalid values in numeric-looking columns
    invalid_numeric_count = 0
    if inferred_dtype == "object":
        # Attempt converting to numeric to find non-convertible strings
        numeric_conversion = pd.to_numeric(col_data, errors="coerce")
        # Count values that were NOT null originally but failed conversion
        invalid_numeric_count = (
            col_data.notna() & numeric_conversion.isna()
        ).sum()

    # Sample non-null values for visual check
    sample_values = col_data.dropna().unique()[:3]
    sample_str = (
        ", ".join(map(str, sample_values)) if len(sample_values) > 0 else "N/A"
    )

    column_audit.append({
        "Column Name": col,
        "Data Type": inferred_dtype,
        "Null Count": null_count,
        "Null %": round(null_pct, 2),
        "Unique Values": unique_count,
        "Invalid Numeric Strings": invalid_numeric_count,
        "Sample Values": sample_str,
    })

# Convert audit list to DataFrame for clean presentation in Jupyter
audit_df = pd.DataFrame(column_audit)

print("=" * 80)
print("2. COLUMN-BY-COLUMN HYGIENE & QUALITY REPORT")
print("=" * 80)
display(audit_df)  # Display formatted interactive HTML table in Jupyter Lab



2. COLUMN-BY-COLUMN HYGIENE & QUALITY REPORT


,Column Name,Data Type,Null Count,Null %,Unique Values,Invalid Numeric Strings,Sample Values
0,STATUS,object,0,0.00,3,1210757,"REVISED, FINAL, CONFIDENTIAL"
1,SER_NBR,int64,0,0.00,27224,0,"9269, 9270, 9271"
2,Series_reference,object,0,0.00,27224,1210757,"HLFQ.S1A1S, HLFQ.S1A2S, HLFQ.S1A3S"
3,Period,float64,0,0.00,157,0,"1986.03, 1986.06, 1986.09"
4,Data_value,float64,67941,5.61,20268,0,"950.0, 948.0, 943.0"
5,UNITS,object,0,0.00,3,1210757,"Number, Percent, number"
6,MAGNTUDE,int64,0,0.00,2,0,"3, 0"
7,Subject,object,0,0.00,1,1210757,Household Labour Force Survey - HLF
8,Group,object,0,0.00,82,1210757,Labour Force Status by Sex: Seasonally Adjuste...
9,Age Group 3 brackets,object,1191269,98.39,4,19488,"Aged 15-24 Years, Aged 25-54 Years, Aged 55 Ye..."


In [4]:
# -----------------------------------------------------------------------------
# 4. SUMMARY RECOMMENDATIONS FOR DATA CLEANING
# -----------------------------------------------------------------------------
print("\n" + "=" * 80)
print("3. AUTOMATED ANOMALY HIGHLIGHTS")
print("=" * 80)

high_null_cols = audit_df[audit_df["Null %"] > 40.0]["Column Name"].tolist()
if high_null_cols:
    print(f"⚠️  High Missing Data Warning (>40% missing): {high_null_cols}")
else:
    print("✅ No columns exceed 40% missing data threshold.")

if duplicate_rows > 0:
    print(
        f"⚠️  Duplicate Warning: Found {duplicate_rows:,} exact matching duplicate rows."
    )
else:
    print("✅ No duplicate rows found.")

cols_with_invalid_nums = audit_df[audit_df["Invalid Numeric Strings"] > 0][
    "Column Name"
].tolist()
if cols_with_invalid_nums:
    print(
        f"⚠️  Mixed Type / Dirty Numeric Warning: Columns containing non-numeric characters: {cols_with_invalid_nums}"
    )
else:
    print("✅ All object columns passed string-to-numeric validation.")


3. AUTOMATED ANOMALY HIGHLIGHTS
⚠️  High Missing Data Warning (>40% missing): ['Age Group 3 brackets', 'Age Group', 'Age Group 6 brackets', 'Disability age breakdown classification', 'Disability status classification', 'Duration of unemployment', 'Employed and Unemployed Persons, Full-Time and Part-Time Status', 'Employment relationship', 'Employment Status', 'Ethnic Single / Combination', 'Ethnic Total Response', 'Formal study status', 'Highest qualification', 'Hours Worked', 'Household Composition', 'Household Labour Force Status', 'Industry ANZSIC06', 'Industry ANZSIC06 Supplementary', 'Job', 'Job tenure', 'Labour force and education status', 'Labour Force Status', 'Labour force/underutilisation classification', 'Main activity', 'Main Job', 'Methods of seeking employment', 'Occupation ANZSCO Level 1', 'Percentage change from previous period and same period previous year', 'Persons employed by job security', 'Persons employed by reason working fewer hours (main job) or away from wor